# 01 — Data Exploration

Standalone EDA reference for the Global Weather Repository dataset.
Covers schema inspection, univariate distributions, correlation matrix,
time-series trend, monthly seasonal boxplots, and STL decomposition.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline

In [ ]:
# Load raw dataset
df = pd.read_csv('../data/raw/GlobalWeatherRepository.csv', low_memory=False)
df.columns = [c.replace(':', '_') for c in df.columns]
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Schema inspection
df.dtypes

In [ ]:
# First few rows
df.head(3)

In [ ]:
# Missing values
missing = df.isnull().sum()
missing[missing > 0].sort_values(ascending=False)

In [ ]:
# Summary statistics
df.describe().T

## Feature Distributions

In [ ]:
num_cols = ['temperature_celsius', 'humidity', 'wind_kph', 'pressure_mb',
            'precip_mm', 'cloud', 'feels_like_celsius', 'visibility_km',
            'uv_index', 'gust_kph']
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    df[col].dropna().hist(bins=50, ax=axes[i], color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(12, 9))
sns.heatmap(corr, mask=mask, ax=ax, cmap='RdBu_r', center=0,
            annot=True, fmt='.2f', annot_kws={'size': 8},
            linewidths=0.5, square=True)
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Temperature Time Series

In [ ]:
df['last_updated'] = pd.to_datetime(df['last_updated'], errors='coerce')
df['date'] = df['last_updated'].dt.date
daily = df.groupby('date')['temperature_celsius'].mean().reset_index()
daily['date'] = pd.to_datetime(daily['date'])

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(daily['date'], daily['temperature_celsius'], alpha=0.5, lw=1, color='steelblue', label='Daily mean')
roll = daily['temperature_celsius'].rolling(30, min_periods=1).mean()
ax.plot(daily['date'], roll, color='darkorange', lw=2, label='30-day MA')
ax.set_title('Global Mean Temperature Over Time', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°C)')
ax.legend()
plt.tight_layout()
plt.show()

## Seasonal Patterns

In [ ]:
daily['month'] = daily['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(12, 5))
data_by_month = [daily[daily['month'] == m]['temperature_celsius'].values for m in range(1, 13)]
ax.boxplot(data_by_month, patch_artist=True)
ax.set_xticklabels(month_names)
ax.set_title('Monthly Temperature Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Temperature (°C)')
plt.tight_layout()
plt.show()

## Top-20 Warmest Countries

In [ ]:
country_temp = df.groupby('country')['temperature_celsius'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
country_temp.head(20).plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.invert_yaxis()
ax.set_title('Top 20 Warmest Countries (Mean Temperature)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean Temperature (°C)')
plt.tight_layout()
plt.show()